# Dual Decoder

In [1]:
import pandas as pd
from pathlib import Path

# plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc

import InterScale as interscale
from InterScale._paths import FIG_PATH
from InterScale.config import load_config
#from InterScale.pl.wandb import load_result_as_df, compute_mean_and_std, summary_df, plot_parameter_space

/usr/local/lib/python3.10/dist-packages/optuna/study/_optimize.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/usr/local/lib/python3.10/dist-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.10/dist-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.10/dist-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.10/dist-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.

## 0. Load paths

In [2]:
import os

# LRZ home
if os.path.exists("/dss/dsshome1/05/di93tig"):
    print('LRZ cluster')
    CLUSTER = 'LRZ'
    BASE_DIR_REPO = "/dss/dsshome1/05/di93tig/1_projects" 
    BASE_DIR_PROJECT = "/dss/dssfs03/tumdss/pn36po/pn36po-dss-0002/di93tig/Projects/A3_InterScale"
elif os.path.exists("/home/icb/francesca.drummer/"):
    print('HPC cluster')
    CLUSTER = 'HPC'
    BASE_DIR_REPO = "/home/icb/francesca.drummer/1-Projects/"
    BASE_DIR_PROJECT = ""
else:
    print('unkown')
    CLUSTER = 'unknown'

LRZ cluster


In [3]:
DUAL_LEGNINI = os.path.join(BASE_DIR_PROJECT, "results/legnini23/legnini23_regr_node_44_GCN_self-attn-transformer_model.ckpt")
DUAL_MELTON = os.path.join(BASE_DIR_PROJECT, "/results/legnini23/legnini23_regr_node_44_GCN_self-attn-transformer_model.ckpt")

In [4]:
CFG_DUAL_LEGNINI = os.path.join(BASE_DIR_REPO, "GT-long-range-niches/src/config_files/Legnini23/legnini23_regr_node_DualCombined_genes.yaml")
CFG_DUAL_MELTON = os.path.join(BASE_DIR_REPO, "GT-long-range-niches/src/config_files/melton25/regr_dual_combined.yaml")

In [5]:
RESULTS_DIR = os.path.join(BASE_DIR_PROJECT, "results")

## 1. Load trained model

In [6]:
cfg = load_config(CFG_DUAL_LEGNINI)
cfg

CfgNode({'wandb': CfgNode({'use': True, 'project_name': 'GTLongRange_Legnini23'}), 'model': CfgNode({'n_embed': 16, 'local_component': CfgNode({'name': 'GCN', 'load': None, 'parameters': CfgNode({'hidden_dim': 256, 'num_layers': 2, 'dropout_local': 0.1})}), 'global_component': CfgNode({'name': 'self-attn-transformer', 'load': None, 'parameters': CfgNode({'n_heads': 4, 'dim_feedforward': 256, 'dropout_global': 0.1, 'activation_func': 'relu', 'num_layers': 2, 'max_seq_len': 4299, 'long_range_attention': False, 'type_gex_embedding': 'PCA'})}), 'save': '/dss/dssfs03/tumdss/pn36po/pn36po-dss-0002/di93tig/Projects/A3_InterScale/results/legnini23/', 'loss': None, 'decoder': CfgNode({'type': 'linear', 'hidden_dims': [256, 128], 'dropout_decoder': 0.1, 'dual_decoder': True})}), 'optim': CfgNode({'accelerator': 'cpu', 'lr': 0.001, 'lr_scheduler': 'CosineWarmupScheduler', 'lr_warmup': 20, 'lr_max_epochs': 100, 'wd': 0.0001, 'loss': 'SmoothL1', 'seed': 44, 'cross_corr': 'cell', 'n_epochs': 100, 'e

In [7]:
adata = sc.read_h5ad(cfg.dataset.h5ad_data)
adata

AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names', 'split'
    var: 'gene_ids', 'feature_types'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

In [8]:
model_name = "legnini23/dual_legnini23_regr_node_44_GCN_self-attn-transformer_model"

In [9]:
interscale.model.CombinedModel._setup_anndata(adata = adata, prediction_task = cfg.dataset.prediction_task, layer_key = cfg.dataset.layer_key, sample_key_list = cfg.dataset.sample_key, prediction_obs =  cfg.dataset.prediction_obs, group_key = cfg.dataset.group_label, view_registry = False)
dual_combined_model = interscale.model.CombinedModel.load(RESULTS_DIR, adata, cfg, model_name = model_name, local_component = True, global_component = True, wandb_save = True)

Loading model from /dss/dssfs03/tumdss/pn36po/pn36po-dss-0002/di93tig/Projects/A3_InterScale/results/legnini23/dual_legnini23_regr_node_44_GCN_self-attn-transformer_modelmodel.pt
Try with .ckpt extension.
Unclear checkpoint format. Attempting remapping anyway.
State dict remapping applied. Source detected: unknown


/usr/local/lib/python3.10/dist-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[log1p_norm] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/dss/dsshome1/05/di93tig/1_projects/GT-long-range-niches/src/InterScale/model/base/_base_model.py:598: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weight

In [10]:
print(dual_combined_model)

In [11]:
dual_combined_model.get_model_output(adata)

Load GEX from .layers/log1p_norm
call new


AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names', 'split', '_scvi_sample_key_0', '_scvi_group_label', '_cls_horizontal', '_cls_vertical'
    var: 'gene_ids', 'feature_types'
    uns: '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'spatial', '_local_emb', '_global_emb', '_attn_matrix'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw', '_y_pred_local', '_y_pred_global'

In [22]:
dual_combined_model.module.state_dict()

OrderedDict([('local_module.decoder.decoder.weight',
              tensor([[ 0.1196,  0.0429,  0.1698,  ..., -0.2071,  0.0729, -0.0760],
                      [ 0.0282,  0.2070,  0.1551,  ..., -0.0220, -0.2039,  0.1772],
                      [ 0.0079, -0.0266,  0.1286,  ..., -0.2244,  0.1426, -0.0448],
                      ...,
                      [-0.2085,  0.0828, -0.1688,  ..., -0.1502,  0.0171, -0.2141],
                      [-0.1938,  0.1130, -0.1366,  ..., -0.1585,  0.1175, -0.0639],
                      [ 0.1866,  0.1411,  0.0433,  ..., -0.2317,  0.1733, -0.0844]])),
             ('local_module.decoder.decoder.bias',
              tensor([ 0.0252, -0.0342, -0.1357,  0.1699,  0.0578,  0.1183, -0.0483, -0.0448,
                       0.1664,  0.1740, -0.1764, -0.1398,  0.0518, -0.0515,  0.2193, -0.1155,
                       0.1637, -0.1640, -0.0704, -0.1735,  0.0581, -0.0866,  0.1838, -0.1220,
                       0.0237,  0.2017, -0.0946,  0.0628, -0.0283,  0.1073,  0.1

In [13]:
local_decoder_weight = dual_combined_model.module.state_dict()['local_module.decoder.decoder.weight']
global_decoder_weight = dual_combined_model.module.state_dict()['global_module.decoder.decoder.weight']

In [21]:
global_decoder_weight

tensor([[ 0.2250, -0.0410, -0.0829,  ..., -0.1766, -0.1399,  0.0829],
        [ 0.2110,  0.0827,  0.1725,  ...,  0.1079,  0.0881, -0.1901],
        [-0.0430,  0.0936, -0.0661,  ...,  0.0900,  0.1811, -0.1519],
        ...,
        [ 0.2774, -0.0060,  0.0858,  ..., -0.2661,  0.2228, -0.1966],
        [-0.0063,  0.0506,  0.1956,  ..., -0.0489,  0.1792,  0.1899],
        [ 0.1550,  0.1903, -0.1749,  ...,  0.2000, -0.0784, -0.0836]])

In [22]:
local_decoder_weight

tensor([[-0.0193,  0.0812,  0.1781,  ...,  0.0723,  0.0973,  0.1413],
        [ 0.0892, -0.0269,  0.3167,  ..., -0.0146,  0.2334,  0.1406],
        [-0.0377, -0.0646, -0.0157,  ..., -0.0517,  0.1967, -0.0939],
        ...,
        [ 0.0868, -0.1574,  0.0256,  ..., -0.1721, -0.0662,  0.0033],
        [-0.0135, -0.1442, -0.0081,  ..., -0.0017,  0.1623, -0.2344],
        [ 0.0827, -0.0095, -0.0231,  ..., -0.0242, -0.0009, -0.1859]])

In [23]:
decoder_weight

tensor([[ 0.1098,  0.1153,  0.1639,  ..., -0.2196,  0.1113, -0.0823],
        [ 0.0175,  0.1058,  0.1489,  ..., -0.0352, -0.1936,  0.1706],
        [-0.0011, -0.0703,  0.1235,  ..., -0.2355,  0.1045, -0.0509],
        ...,
        [-0.2187,  0.0319, -0.1746,  ..., -0.1629,  0.0531, -0.2206],
        [-0.2029,  0.0966, -0.1434,  ..., -0.1697,  0.0663, -0.0703],
        [ 0.1976,  0.1966,  0.0488,  ..., -0.2181,  0.2183, -0.0778]])